# Experimento Entrenamiento Catboost

**Descripción**: Realizo el entrenamiento con el algoritmo de boosting catboost, este algoritmo es seleccionado ya que presenta algunas ventajas importantes:
- Manejo nativo de datos nulos, no requiere imputación.
- Manejo nativo de variables categóricas, importante en nuestro caso ya que contamos con varias de este estilo.
- Al ser un algoritmo de árboles de decisión es más resistente a outliers por lo que no requiere escalamiento.  

Se setea la semilla 42 para poder reproducir resultados.

Se loguean los resultados de los experimentos con MLflow.

---

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from catboost import CatBoostClassifier, Pool, cv
from sklearn.metrics import (classification_report, roc_auc_score,
                              accuracy_score, precision_score,
                              recall_score, f1_score)
import plotly.express as px
from plotly import graph_objects as go
import mlflow
import mlflow.catboost
import optuna
import numpy as np


In [2]:
# --- Carga de datos ---

X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_test  = pd.read_parquet('../data/processed/X_test.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')
y_test  = pd.read_parquet('../data/processed/y_test.parquet')

# --- Encodear target ---
y_train_enc = (y_train.iloc[:, 0] == 'new').astype(int)
y_test_enc  = (y_test.iloc[:, 0]  == 'new').astype(int)

# --- Variables categóricas para pasarle al modelo ---
cat_features = ['category_id', 'city', 'warranty', 'listing_type_id', 
                'shipping_mode', 'buying_mode', 'state']

mlflow.set_tracking_uri('../models/mlruns')
mlflow.set_experiment('CatBoost')

# --- Baseline CatBoost sin tuning ---
with mlflow.start_run(run_name='catboost_baseline'):

    mlflow.log_params({
        'model':                'CatBoostClassifier',
        'iterations':           1000,
        'random_seed':          42,
        'eval_metric':          'AUC',
        'early_stopping_rounds': 50,
    })

    model = CatBoostClassifier(iterations=1000, random_seed=42,
                                eval_metric='AUC', verbose=100)
    model.fit(X_train, y_train_enc,
              cat_features=cat_features,
              eval_set=(X_test, y_test_enc),
              early_stopping_rounds=50)

    y_pred       = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    mlflow.log_metrics({
        'test_accuracy':       accuracy_score(y_test_enc, y_pred),
        'test_auc':            roc_auc_score(y_test_enc, y_pred_proba),
        'test_precision_used': precision_score(y_test_enc, y_pred, pos_label=0),
        'test_recall_used':    recall_score(y_test_enc, y_pred, pos_label=0),
        'test_f1_used':        f1_score(y_test_enc, y_pred, pos_label=0),
        'test_precision_new':  precision_score(y_test_enc, y_pred, pos_label=1),
        'test_recall_new':     recall_score(y_test_enc, y_pred, pos_label=1),
    })

    mlflow.catboost.log_model(model, name='model')
    print(f"Run ID: {mlflow.active_run().info.run_id}")

print(classification_report(y_test_enc, y_pred, target_names=['used', 'new']))
print(f"AUC-ROC: {roc_auc_score(y_test_enc, y_pred_proba):.4f}")

/Users/patriciogarbino/.pyenv/versions/meli-challenge/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Learning rate set to 0.096296
0:	test: 0.9087480	best: 0.9087480 (0)	total: 104ms	remaining: 1m 43s
100:	test: 0.9679115	best: 0.9679142 (99)	total: 3.67s	remaining: 32.6s
200:	test: 0.9703854	best: 0.9703854 (200)	total: 6.84s	remaining: 27.2s
300:	test: 0.9715124	best: 0.9715124 (300)	total: 10s	remaining: 23.2s
400:	test: 0.9722746	best: 0.9722746 (399)	total: 13.3s	remaining: 19.9s
500:	test: 0.9727815	best: 0.9727949 (494)	total: 16.7s	remaining: 16.6s
600:	test: 0.9730335	best: 0.9730350 (598)	total: 20.2s	remaining: 13.4s
700:	test: 0.9731886	best: 0.9732158 (681)	total: 23.7s	remaining: 10.1s
800:	test: 0.9733061	best: 0.9733233 (791)	total: 27.2s	remaining: 6.75s
900:	test: 0.9735166	best: 0.9735166 (900)	total: 30.5s	remaining: 3.35s
999:	test: 0.9735973	best: 0.9736090 (973)	total: 33.9s	remaining: 0us

bestTest = 0.9736089925
bestIteration = 973

Shrink model to first 974 iterations.
Run ID: d2b7349f283342b3a493992479647437
              precision    recall  f1-score   supp

In [3]:
# --- Feature Importance ---

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=True)

fig = px.bar(
    feature_importance,
    x='importance',
    y='feature',
    orientation='h',
    title='Feature Importance — CatBoost',
    template='plotly_white'
)
fig.update_layout(height=700, width=800)
fig.show()

In [4]:
# --- Curvas de entrenamiento ---

evals_result = model.get_evals_result()

train_auc = evals_result['learn']['Logloss']
val_auc   = evals_result['validation']['Logloss']

fig = go.Figure()
fig.add_trace(go.Scatter(y=train_auc, name='Train Logloss', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=val_auc,   name='Val Logloss',   line=dict(color='red')))

fig.update_layout(
    title='Curvas de entrenamiento — CatBoost',
    xaxis_title='Iteración',
    yaxis_title='Logloss',
    template='plotly_white',
    height=500, width=900
)
fig.show()

### Análisis de los resultados

El modelo alcanza un accuracy de **0.92**, quedando 6pp sobre el mínimo requerido
de 0.86 y 10pp sobre el baseline. El AUC-ROC de **0.97** indica que el modelo tiene excelente capacidad
discernir entre las dos clases

La **precisión de "used" es 0.90**, al igual que comenté en el ejercicio del baseline, esta métrica es la que considero más importante para el modelo ya que creo que a nivel de negocio lo peor que puede hacer el algoritmo es decir que un producto usado es nuevo. Si bien es un buen número el algoritmo lo hace mejor en la clase **new** podríamos mover el threshol de decisión para favorecer a nuestra clase objetivo.

El balance entre clases es adecuado — precisión y recall similares entre `new` y `used`
indican que el modelo no está sesgado hacia ninguna clase, lo cual es consistente con el
balance del dataset (~54% new, ~46% used).

Existen dos variables que no poseen impacto significativo al modelo `deal_ids` y `official_store_id` probaremos en el siguiente experimento eliminarlas.

Las curvas de entrenamiento muestra comienzo de overfitting en la iteración 600 aunque no es demasiado, se puede ajustar el early stopping un poco.

### 1.2 Eliminación de columnas con baja importancia y ajuste de early stopping

In [5]:
# --- Selección de variables ---

cols =['warranty', 'listing_type_id', 'price', 'buying_mode',
       'category_id', 'automatic_relist', 'video_id',
       'initial_quantity', 'sold_quantity', 'state', 'city',
       'shipping_local_pick_up', 'shipping_free', 'shipping_mode',
       'non_mercado_pago_payment_methods_count', 'variations_count',
       'tags_count', 'attributes_count', 'pictures_count',
       'pictures_max_area']

# --- Entrenamiento ---

with mlflow.start_run(run_name='catboost_selected_features'):

    mlflow.log_params({
        'model':                'CatBoostClassifier',
        'iterations':           1000,
        'random_seed':          42,
        'eval_metric':          'AUC',
        'early_stopping_rounds': 30,
    })

    model = CatBoostClassifier(iterations=1000, random_seed=42,
                                eval_metric='AUC', verbose=100)
    model.fit(X_train[cols], y_train_enc,
              cat_features=cat_features,
              eval_set=(X_test[cols], y_test_enc),
              early_stopping_rounds=30) # Ajuste de early stopping a 30

    y_pred       = model.predict(X_test[cols])
    y_pred_proba = model.predict_proba(X_test[cols])[:, 1]

    mlflow.log_metrics({
        'test_accuracy':       accuracy_score(y_test_enc, y_pred),
        'test_auc':            roc_auc_score(y_test_enc, y_pred_proba),
        'test_precision_used': precision_score(y_test_enc, y_pred, pos_label=0),
        'test_recall_used':    recall_score(y_test_enc, y_pred, pos_label=0),
        'test_f1_used':        f1_score(y_test_enc, y_pred, pos_label=0),
        'test_precision_new':  precision_score(y_test_enc, y_pred, pos_label=1),
        'test_recall_new':     recall_score(y_test_enc, y_pred, pos_label=1),
    })

    mlflow.catboost.log_model(model, name='model')
    print(f"Run ID: {mlflow.active_run().info.run_id}")

print(classification_report(y_test_enc, y_pred, target_names=['used', 'new']))
print(f"AUC-ROC: {roc_auc_score(y_test_enc, y_pred_proba):.4f}")

Learning rate set to 0.096296
0:	test: 0.9063461	best: 0.9063461 (0)	total: 48.4ms	remaining: 48.4s
100:	test: 0.9679182	best: 0.9679182 (100)	total: 4.24s	remaining: 37.8s
200:	test: 0.9706863	best: 0.9706863 (200)	total: 7.96s	remaining: 31.6s
300:	test: 0.9716285	best: 0.9716285 (300)	total: 11.4s	remaining: 26.4s
400:	test: 0.9722845	best: 0.9722865 (399)	total: 15s	remaining: 22.3s
500:	test: 0.9726238	best: 0.9726291 (499)	total: 18.8s	remaining: 18.7s
600:	test: 0.9729525	best: 0.9729672 (595)	total: 22.8s	remaining: 15.1s
700:	test: 0.9733319	best: 0.9733371 (699)	total: 26.5s	remaining: 11.3s
800:	test: 0.9735781	best: 0.9735781 (800)	total: 30.9s	remaining: 7.66s
900:	test: 0.9737396	best: 0.9737438 (897)	total: 35.8s	remaining: 3.93s
Stopped by overfitting detector  (30 iterations wait)

bestTest = 0.9738175274
bestIteration = 950

Shrink model to first 951 iterations.
Run ID: 69a3bd531d734c62b69584adc6d6406a
              precision    recall  f1-score   support

        use

In [6]:
# --- Feature Importance ---

feature_importance = pd.DataFrame({
    'feature': X_train[cols].columns,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=True)

fig = px.bar(
    feature_importance,
    x='importance',
    y='feature',
    orientation='h',
    title='Feature Importance — CatBoost',
    template='plotly_white'
)
fig.update_layout(height=700, width=800)
fig.show()

In [7]:
# --- Curvas de entrenamiento ---

evals_result = model.get_evals_result()

train_auc = evals_result['learn']['Logloss']
val_auc   = evals_result['validation']['Logloss']

fig = go.Figure()
fig.add_trace(go.Scatter(y=train_auc, name='Train Logloss', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=val_auc,   name='Val Logloss',   line=dict(color='red')))

fig.update_layout(
    title='Curvas de entrenamiento — CatBoost',
    xaxis_title='Iteración',
    yaxis_title='Logloss',
    template='plotly_white',
    height=500, width=900
)
fig.show()

### Análisis de los resultados

El cambio en el early stopping a 30 iteraciones no tiene impacto en las curvas de entrenamiento, se debería reducir más.

El remover las features produce un poco de pérdida de performance por lo que podria mantenerlas si no es costoso generarlas.

### 1.3 Optimización Bayesiana con Optuna

Procedo a realizar una búsqueda bayessiana de hyperparámetros usando optuna y buscando optimizar nuestra clase objetivo **precision de la case used**.

La validación se hace con cross validation en train para evitar leakage con Test.

In [8]:
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 0, 1),
        'iterations': 1000,
        'loss_function': 'Logloss',
        'eval_metric': 'Precision',   # precisión de la clase positiva (1 por defecto)
        'random_seed': 42,
        'verbose': 0
    }

    pool = Pool(X_train, y_train_enc, cat_features=cat_features)
    
    cv_results = cv(
        pool,
        params,
        fold_count=5,
        early_stopping_rounds=50,
        stratified=True,
        as_pandas=True
    )
    
    return cv_results['test-Precision-mean'].iloc[-1]

sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best precision:", study.best_value)

[I 2026-03-18 21:17:29,734] A new study created in memory with name: no-name-2d4d6aa6-77f1-440e-97fa-642a4c560de3


  0%|          | 0/50 [00:00<?, ?it/s]

Training on fold [0/5]

bestTest = 0.9244861008
bestIteration = 172

Training on fold [1/5]

bestTest = 0.9255045084
bestIteration = 195

Training on fold [2/5]

bestTest = 0.9286247733
bestIteration = 217

Training on fold [3/5]

bestTest = 0.9330755183
bestIteration = 220

Training on fold [4/5]

bestTest = 0.9277712579
bestIteration = 393

[I 2026-03-18 21:19:25,708] Trial 0 finished with value: 0.927480744961259 and parameters: {'learning_rate': 0.03574712922600244, 'depth': 10, 'l2_leaf_reg': 7.587945476302646, 'bagging_temperature': 0.5986584841970366, 'random_strength': 0.15601864044243652}. Best is trial 0 with value: 0.927480744961259.
Training on fold [0/5]

bestTest = 0.9025829437
bestIteration = 23

Training on fold [1/5]

bestTest = 0.9178067435
bestIteration = 276

Training on fold [2/5]

bestTest = 0.9212471874
bestIteration = 359

Training on fold [3/5]

bestTest = 0.9233415762
bestIteration = 338

Training on fold [4/5]

bestTest = 0.9024948251
bestIteration = 39

[I 2

In [9]:
# --- Reentrenar mejor modelo y loguear en MLflow ---

mlflow.set_experiment('CatBoost')

with mlflow.start_run(run_name='catboost_optuna'):

    mlflow.log_params({
        **study.best_params,
        'iterations_max': 1000,
        'early_stopping_rounds': 50,
        'n_trials': 50,
        'optuna_objective': 'precision_used',
        'cv_folds': 5,           
    })

    y_train_inv = 1 - y_train_enc
    y_test_inv  = 1 - y_test_enc

    best_model = CatBoostClassifier(
        **study.best_params,
        iterations=1000,
        random_seed=42,
        verbose=100,
        loss_function='Logloss',
        eval_metric='Precision',
        custom_metric=['AUC'],
    )

    best_model.fit(
        X_train, y_train_inv,
        cat_features=cat_features,
        eval_set=(X_test, y_test_inv),  
        early_stopping_rounds=50,
        plot=False
    )

    # Evaluar en test —
    y_pred      = best_model.predict(X_test)
    y_pred_orig = 1 - y_pred

    y_proba_used = best_model.predict_proba(X_test)[:, 1]

    mlflow.log_metrics({
        'test_accuracy':       accuracy_score(y_test_enc, y_pred_orig),
        'test_auc':            roc_auc_score(y_test_enc, y_proba_used),
        'test_precision_used': precision_score(y_test_enc, y_pred_orig, pos_label=0),
        'test_recall_used':    recall_score(y_test_enc, y_pred_orig, pos_label=0),
        'test_f1_used':        f1_score(y_test_enc, y_pred_orig, pos_label=0),
        'test_precision_new':  precision_score(y_test_enc, y_pred_orig, pos_label=1),
        'test_recall_new':     recall_score(y_test_enc, y_pred_orig, pos_label=1),
        'val_precision_used':  study.best_value,
    })

    mlflow.catboost.log_model(best_model, artifact_path='model')
    best_model.save_model('../models/catboost_model.cbm')
    mlflow.log_artifact('../models/catboost_model.cbm')
    print(f"Run ID: {mlflow.active_run().info.run_id}")

print(classification_report(y_test_enc, y_pred_orig, target_names=['used', 'new']))
print(f"AUC-ROC: {roc_auc_score(y_test_enc, y_proba_used):.4f}")

0:	learn: 0.7958548	test: 0.7907576	best: 0.7907576 (0)	total: 213ms	remaining: 3m 33s
100:	learn: 0.8872067	test: 0.8878172	best: 0.8886788 (95)	total: 4.94s	remaining: 44s
200:	learn: 0.8968635	test: 0.8918406	best: 0.8927066 (174)	total: 9.86s	remaining: 39.2s


2026/03/18 22:15:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8927065767
bestIteration = 174

Shrink model to first 175 iterations.
Run ID: c9c09fac265d4aaaa308c89e65aeef2a
              precision    recall  f1-score   support

        used       0.89      0.92      0.91      4594
         new       0.93      0.91      0.92      5406

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000

AUC-ROC: 0.0272


In [10]:
# --- Curvas de entrenamiento ---

evals_result = best_model.get_evals_result()

train_auc = evals_result['learn']['Logloss']
val_auc   = evals_result['validation']['Logloss']

fig = go.Figure()
fig.add_trace(go.Scatter(y=train_auc, name='Train Logloss', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=val_auc,   name='Val Logloss',   line=dict(color='red')))

fig.update_layout(
    title='Curvas de entrenamiento — CatBoost',
    xaxis_title='Iteración',
    yaxis_title='Logloss',
    template='plotly_white',
    height=500, width=900
)
fig.show()

### Análisis de los resultados

La optimización con Optuna no logró mejorar las métricas obtenidas con nuestra primer modelo, lo que sugiere que nuestro modelo ya se encuentra cercano al valor óptimo que podemos alcanzar con estos datos y features.

### 1.4 Ajuste de Threshold

Dado que pretendemos mejorar la precisión de la clase "used", podemos ajustar el threshold de clasificación para favorecer esa clase. Por defecto, el threshold es 0.5, pero podríamos probar valores más bajos para aumentar la cantidad de predicciones "used" (clase 0) y así mejorar su precisión.

In [15]:
import numpy as np

print(classification_report(y_test_enc, 
                            np.where(y_pred_proba > 0.35, 1, 0),
                            target_names=['used', 'new']))


              precision    recall  f1-score   support

        used       0.92      0.88      0.90      4594
         new       0.90      0.94      0.92      5406

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000



### Análisis de los resultados

Al ubicar nuestro corte de probabilidades en **0.3** logramos mejorar la precisión de la clase used sin sacrificar accuracy y manteniendo el f1 de ambas clases.

In [44]:
# guardar
best_model.save_model('../models/catboost_model.cbm')